In [ ]:
import os
import pandas as pd
from lxml import etree

In [ ]:
def minimal_fix(xml_text):

    xml_text = xml_text.replace("& ", "&amp; ")
    xml_text = xml_text.replace("&\n", "&amp;\n")


    wrapped = "<root>\n" + xml_text + "\n</root>"


    parser = etree.XMLParser(recover=True)
    root = etree.fromstring(wrapped.encode(), parser=parser)

    return root


###########################################################
# 2. List of all corpus files
###########################################################

paths = [
    "F1_fully_annotated.txt",
    "F2_fully_annotated.txt",
    "GC1_1_fully_annotated.txt",
    "GC1_2_fully_annotated.txt",
    "M1_fully_annotated.txt",
    "M2_fully_annotated.txt",
    "M7_1_fully_annotated.txt",
    "M7_2_fully_annotated.txt",
    "M7_3_fully_annotated.txt"
]


In [ ]:

###########################################################
# 3. Build CSV rows
###########################################################

rows = []

for path in paths:
    txt = open(path).read()
    root = minimal_fix(txt)

    # --- Extract session data ---
    session_elem = root.find("session")
    session_attrs = session_elem.attrib if session_elem is not None else {}

    # FIRST ROW FOR EACH FILE:
    rows.append({
        "session": str(session_attrs),
        "sound": "",
        "TURN ID": "",
        "turn length": "",
        "Intro": "",
        "speaker code": "",
        "speaker nationality": "",
        "speaker languages": "",
        "num langs": "",
        "Transcript": "",
        "misunderstanding": "",
        "mis_type": "",
        "translanguaging": "",
        "trans_type_count": "",
        "repair": "",
        "repair_count": "",
        "interactional_move": "",
        "imove_type": "",
        "content": ""
    })

    ###########################################################
    # 4. Extract sound and turn elements in order
    ###########################################################

    for elem in root.iter():

        # -------------------------
        # SOUND ROW
        # -------------------------
        if elem.tag == "sound":
            rows.append({
                "session": "",
                "sound": elem.text.strip() if elem.text else "",
                "TURN ID": "",
                "turn length": "",
                "Intro": "",
                "speaker code": "",
                "speaker nationality": "",
                "speaker languages": "",
                "num langs": "",
                "Transcript": "",
                "misunderstanding": "",
                "mis_type": "",
                "translanguaging": "",
                "trans_type_count": "",
                "repair": "",
                "repair_count": "",
                "interactional_move": "",
                "imove_type": "",
                "content": ""
            })

        # -------------------------
        # TURN ROW
        # -------------------------
        if elem.tag == "turn":
            turn = elem

            # speaker info
            sp = turn.find("speaker")
            code = sp.get("code") if sp is not None else ""
            nat = sp.get("nationality") if sp is not None else ""
            langs = sp.get("language") if sp is not None else ""
            num_langs = len(langs.split(",")) if langs else ""

            # intro yes/no
            intro_flag = "yes" if turn.find("intro") is not None else "no"

            # nested annotations
            trans_elems = turn.findall(".//translanguaging")
            repair_elems = turn.findall(".//repair")
            im_elems = turn.findall(".//interactional_move")

            # misunderstanding: check parent
            parent = turn.getparent()
            mis_flag = (parent.tag == "misunderstanding")
            mis_type = parent.get("type") if mis_flag else "N/A"

            # transcript = only speaker text
            transcript = "".join(sp.itertext()).strip() if sp is not None else ""

            # content = whole turn text (flatten)
            content = "".join(turn.itertext()).strip()

            # write row
            rows.append({
                "session": "",
                "sound": "",
                "TURN ID": turn.get("ID", ""),
                "turn length": turn.get("Length", ""),
                "Intro": intro_flag,
                "speaker code": code,
                "speaker nationality": nat,
                "speaker languages": langs,
                "num langs": num_langs,
                "Transcript": transcript,
                "misunderstanding": "yes" if mis_flag else "no",
                "mis_type": mis_type,
                "translanguaging": "yes" if trans_elems else "no",
                "trans_type_count": len(trans_elems) if trans_elems else "N/A",
                "repair": "yes" if repair_elems else "no",
                "repair_count": len(repair_elems) if repair_elems else "N/A",
                "interactional_move": "yes" if im_elems else "no",
                "imove_type": ",".join([e.get("type", "") for e in im_elems]) if im_elems else "N/A",
                "content": content
            })


###########################################################
# 5. Convert to CSV
###########################################################

df = pd.DataFrame(rows)
output_path = base + "nomad_corpus.csv"
df.to_csv(output_path, index=False)

print("CSV generated:", output_path)